                 TRAINING
                    │
                    ▼
             train_one_epoch()
                    │
             weights updated
                    │
                    ▼
                 VALIDATION
                    │
                    ▼
                validate()
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
    validation loss       no weights changed

In [1]:
import torch

from src.training.validate import validate
from src.training.losses import build_loss
from src.models.baseline_cnn import BaselineCNN

model = BaselineCNN()

print(model)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): AdaptiveAvgPool2d(output_size=(1, 1))
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [2]:
from src.data.labels import load_labels
from src.utils.config import TRAIN_LABELS

labels, stats = load_labels(TRAIN_LABELS)

print("Total patients:", len(labels))
print("Stats:", stats)

Total patients: 26684
Stats: {'num_patients': 26684, 'positive_cases': 6012, 'negative_cases': 20672, 'total_boxes': 9555}


In [3]:
from src.data.split_data import split_patients

train_ids, val_ids, test_ids = split_patients(
    labels=labels
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 18678
Validation: 4002
Test: 4004


In [4]:
from src.data.dataset import RSNADataset
from src.data.dataloader import build_dataloader
from src.utils.config import TRAIN_IMAGES

val_dataset = RSNADataset(
    patient_ids = val_ids,
    labels=labels,
    image_dir=TRAIN_IMAGES
)

val_loader = build_dataloader(
    dataset=val_dataset,
    batch_size=8,
    shuffle=False
)

print("Validation samples:", len(val_dataset))
print("Validation batches:", len(val_loader))

print(type(val_ids))
print(len(val_ids))
print(type(labels))

Validation samples: 4002
Validation batches: 501
<class 'list'>
4002
<class 'dict'>


In [5]:
batch = next(iter(val_loader))

print("Image type:", type(batch["image"]))
print("Image shape:", batch["image"].shape)
print("Image dtype:", batch["image"].dtype)
print("Target type:", type(batch["target"]))
print("Patient IDs:", batch["patient_id"][:3])

Image type: <class 'torch.Tensor'>
Image shape: torch.Size([8, 1, 512, 512])
Image dtype: torch.float32
Target type: <class 'torch.Tensor'>
Patient IDs: ['eb258f84-16e6-4927-8383-c576e8e242d3', '60be0810-9065-4e17-9828-b89b3a383b6a', '69df60a0-cb3d-4da5-9a3b-9d8c36e2c113']


In [6]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Device : ",device)

model = model.to(device)

criterion = build_loss()

val_loss = validate(
    model = model,
    dataloader=val_loader,
    criterion=criterion,
    device=device
)

print(f"validation loss : {val_loss : .4f}")

Device :  cuda
validation loss :  0.6724


In [7]:
import time

start = time.time()

batch = next(iter(val_loader))

print("Batch loading time:", time.time() - start)

images = batch["image"].to(device)

start = time.time()

with torch.no_grad():
    output = model(images)

print("GPU inference time:", time.time() - start)

Batch loading time: 0.41556715965270996
GPU inference time: 0.0011594295501708984
